# Method Validation
### Roadrunner propulsion reverse-engineering study · notebook 4 of 13

**Purpose.** There is no ground truth for the vehicle, so its results can never be
checked directly. What *can* be checked is the method (ADR-0012). This notebook does
two things with the purchasable-engine catalogue, whose datasheets we do have:

1. **Anchor** a pyCycle turbojet model to each datasheet and see whether a simple
   cycle can reproduce what the vendor publishes.
2. **Predict each engine blind**: compute its specific thrust and specific fuel
   consumption from technology levels taken from the *other* engines, and compare
   with its datasheet.

> *How well does the cycle model recover the specific thrust and SFC of an engine it
> has not seen?*

Specific thrust and SFC are intensive, so this tests the cycle model itself. Engine
size, which enters only as air mass flow, is not predicted here (ADR-0024).

---
## 0 · Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

from roadrunner_re import load_engine_catalogue, reports
from roadrunner_re import technology as tech
from roadrunner_re import validation as val
from roadrunner_re.cycle import load_cycle_settings
from roadrunner_re.notebook import nb_setup
from roadrunner_re.plots.validation import blind_predictions

paths = nb_setup()
engines = load_engine_catalogue()
metrics = tech.catalogue_metrics(engines, tech.load_fuel_density().value, tech.load_gravity())

---
## 1 · The cycle model and what it has to assume

A single-spool turbojet in pyCycle (ADR-0015): inlet, centrifugal compressor,
combustor, single-stage turbine, convergent nozzle; stations numbered per ARP755.
Thermodynamics are pyCycle's tabular Jet-A data. Vendors publish thrust, air flow,
fuel flow and sometimes pressure ratio and exhaust temperature. Everything else is
assumed in `config/cycle.yaml` — **do not hard-code numbers in this notebook.**

In [ ]:
settings = load_cycle_settings()
for name, value in settings.nominal_losses().items():
    u = getattr(settings, name)
    print(f"{name:30s}{value:7.3f}   bounds {u.low:.2f} to {u.high:.2f}")
u = settings.pressure_ratio_when_unpublished
print(f"{'pressure ratio if unpublished':30s}{u.value:7.3f}   bounds {u.low:.2f} to {u.high:.2f}")

---
## 2 · Anchoring the model to each datasheet

### Theory

At sea-level static conditions, with the pressure ratio $\pi_c$ from the datasheet
and the component losses assumed, the specific thrust $F_s = F/\dot m_a$ depends on
one remaining unknown, the turbine inlet temperature $T_{t4}$. The solver finds the
$T_{t4}$ that gives the published $F_s$.

The model burns its fuel completely, so its SFC is an ideal one. The ratio

$$\eta_b = \frac{\mathrm{SFC}_\text{ideal}}{\mathrm{SFC}_\text{published}}$$

is reported as a **lumped combustion efficiency**: it absorbs real combustion
inefficiency, the oil these engines burn with their fuel, and any error in the assumed
losses.

The published exhaust gas temperature (EGT) is **not** used in the match. Comparing
it with the model's turbine exit temperature $T_{t5}$ is therefore an independent
check. In the table, $F_s$ is in N s/kg and SFC in kg/(N h).

In [ ]:
anchors = val.anchor_catalogue(engines, metrics, settings)
print(reports.anchor_table(anchors))
print("\n* pressure ratio not published; catalogue value assumed")

Engines with no published air mass flow have no specific thrust and cannot be
anchored; they are absent from the table. Three things stand out:

- **Specific thrust is nearly constant**, about 590 to 635 N s/kg, apart from one
  small core near 505. The anchored turbine inlet temperatures follow it.
- **Combustion efficiency falls with engine size.** The largest anchored engines sit
  near 0.94 to 0.99; the smallest need 0.75 to 0.83 to account for their published
  SFC.
- **The model's turbine exit temperature does not match the published EGT**, and
  not even in a consistent direction. A one-dimensional cycle cannot reconcile a
  vendor's specific thrust, SFC and EGT all at once. A single probe in a small
  exhaust is not a mass-averaged total temperature, so the published EGT is weak
  evidence; but the disagreement is recorded as a standing finding rather than
  explained away.


---
## 3 · Predicting each engine blind

For each engine in turn, nothing is taken from its own datasheet. From the *other*
anchored engines come two technology ranges, turbine inlet temperature and combustion
efficiency; the pressure ratio range is the catalogue's. Then

$$F_s = F_s(T_{t4}, \pi_c), \qquad
\mathrm{SFC} = \frac{\mathrm{SFC}_\text{ideal}(T_{t4}, \pi_c)}{\eta_b}$$

from pyCycle at the nominal point and at the corners of the technology ranges.
Variants of one core are left out together. A prediction **passes** if the published
value lies inside the predicted interval and the interval is no wider than the limit
set beforehand in `config/validation.yaml`.

In [ ]:
limits = val.load_validation_limits()
predictions = val.leave_one_out(engines, metrics, anchors, settings, limits)

print(reports.blind_prediction_table(predictions, "specific_thrust_N_s_kg", unit="[N s/kg]"), end="\n\n")
print(reports.blind_prediction_table(predictions, "sfc_kg_N_s", tech.SECONDS_PER_HOUR, "[kg/(N h)]"))

In [ ]:
fig = blind_predictions(predictions, paths.figures / "method_validation.png")

In [ ]:
for quantity, count in val.pass_counts(predictions).items():
    print(f"{quantity:22s} {count['passed']} of {count['of']} pass   (width limit {limits[quantity]:.0%} of nominal)")

---
## 4 · Handoff

In [ ]:
handoff = val.write_validation_handoff(val.validation_handoff(anchors, predictions), paths.out / "validation.yaml")
print(f"wrote {handoff.relative_to(paths.repo)}")

---
## Findings

- **Specific thrust of the mainstream engines is recovered to within about 6 %** at
  the nominal. The published value lies inside the predicted interval for five of
  eight engines.
- **Three specific-thrust predictions fail**, and they are the catalogue's two
  extremes: the engine with the lowest specific thrust (two variants of one core,
  about 20 % below the prediction) and the one with the highest (just above the
  interval). Leaving an extreme out removes it from the range that is supposed to
  contain it; that is the test working, and it shows the catalogue is too small for
  its own range to bound the next engine.
- **SFC is recovered within about 18 %**, with the published value inside the
  interval for all six engines that publish one.
- **Standing finding:** the model's turbine exit temperature differs from published
  EGT by up to about 130 K in either direction.
- **Standing finding:** the anchored turbine inlet temperatures depend on the assumed
  compressor and turbine efficiencies, which no vendor publishes. They are plausible
  for uncooled small turbines but they are not measurements.
- These errors are the ones to quote next to any specific thrust or SFC the cycle
  model produces for the vehicle study.
